# AIML 420 - Assignment 3

* author: Jason Pollock
* email: jason@pollock.ca
* email: pollocjaso@myvuw.ac.nz

In [7]:
from IPython.display import display
from IPython.display import Markdown

def report(string):
    display(Markdown(string))

class Table:
    def header(self, *fields):
        self.doc = "|"
        for field in fields:
            self.doc += f" {field} |"
        self.doc += "\n"
        self.doc += "|"
        for field in fields:
            self.doc += " --- |"
        self.doc += "\n"
        return self

    def __init__(self):
        self.doc = ""

    def row(self, *values):
        self.doc += "|"
        for value in values:
            self.doc += f" {value} |"
        self.doc += "\n"
        return self

    def report(self):
        report(self.doc)



## Part 1: Job Scheduling

### Question 1: Earliest Starting Time

Process order:

* T1 = Process(O11, M1, t1, 50)
* T2 = Process(O21, M2, t2, 30)
* T3 = Process(O31, M1, t3, 40)
* T4 = Process(O12, M2, t4, 25)
* T5 = Process(O22, M1, t5, 35)
* T6 = Process(O32, M2, t6, 20)

I found myself running the algorithm in a Latex table, it's easier to write the code and debug.

In [17]:
m1_next_available = 0
m2_next_available = 0

t1 = max(0, 0, m1_next_available)
m1_next_available = t1 + 50
t1_completion = m1_next_available

t2 = max(t1, 10, m2_next_available)
m2_next_available = t2 + 30
t2_completion = m2_next_available

t3 = max(t2, 20, m1_next_available)
m1_next_available = t3 + 40
t3_completion = m1_next_available

t4 = max(t3, t1_completion, m2_next_available)
m2_next_available = t4 + 25
t4_completion = m2_next_available

t5 = max(t4, t2_completion, m1_next_available)
m1_next_available = t5 + 35
t5_completion = m1_next_available

t6 = max(t5, t3_completion, m2_next_available)
m2_next_available = t6 + 20
t6_completion = m2_next_available

report("Operation Start Times:")
table = Table()
table.header("Operation", "Start Time")
table.row("T1", t1)
table.row("T2", t2)
table.row("T3", t3)
table.row("T4", t4)
table.row("T5", t5)
table.row("T6", t6)

table.report()


Operation Start Times:

| Operation | Start Time |
| --- | --- |
| T1 | 0 |
| T2 | 10 |
| T3 | 50 |
| T4 | 50 |
| T5 | 90 |
| T6 | 90 |


### Question 2 Completion Time and Makespan

In [18]:
report("Job Completion Times:")
table = Table()
table.header("Job", "Completion Time")
table.row("J1", t4_completion)
table.row("J2", t5_completion)
table.row("J3", t6_completion)
table.report()

report(f"Makespan: {max(t4_completion, t5_completion, t6_completion)}")

Job Completion Times:

| Job | Completion Time |
| --- | --- |
| J1 | 75 |
| J2 | 125 |
| J3 | 110 |


Makespan: 125

### Question 3: Shortest Processing Time SPT Dispatch

Assumption: The machines are not interchangeable, so the only thing changing is the T1<=T2<=T3 requirement.

Solving in code, there would be 3 heaps, pending, ready_machine1 and ready_machine2. At each time step, check if any tasks become ready, moving them to the heap. When the timestamp exceeds a machine's completion time, pop the top off the heap and start running it.

Could enhance this for "real" times by taking the max of the pending, and machine execution timestamps and processing that event.

The task flow below was achieved by running that algorithm iteratively and modifying to code to reflect completed tasks.

In [16]:
def process(start, length, machine_next_available):
    task_start = max(start, machine_next_available)
    task_end = task_start + length
    return (task_start, task_end, task_end)

m1_next_available = 0
m2_next_available = 0

report("Task Processing Order Times:")
table = Table()
table.header("Process")

t1_start, t1_end, m1_next_available = process(0, 50, m1_next_available) # O11
table.row(f"process(O11, M1, {t1_start})")

t2_start, t2_end, m2_next_available = process(10,30, m2_next_available) # O21
table.row(f"process(O21, M2, {t2_start})")

t3_start, t3_end, m2_next_available = process(t1_end, 25, m2_next_available) # O12
table.row(f"process(O12, M2, {t3_start})")

t4_start, t4_end, m1_next_available = process(t2_end, 35, m1_next_available) # O22
table.row(f"process(O22, M1, {t4_start})")

t5_start, t5_end, m1_next_available = process(20, 40, m1_next_available) # O31
table.row(f"process(O31, M1, {t5_start})")

t6_start, t6_end, m2_next_available = process(t5_end, 20, m2_next_available) # O31
table.row(f"process(O32, M2, {t6_start})")

table.report()

report("<br>")
report("Job Completion Times")
table = Table()
table.header("Job", "Completion Time")
table.row("J1", t3_end)
table.row("J2", t4_end)
table.row("J3", t6_end)
table.report()

report(f"Makespan: {max(t3_end, t4_end, t6_end)}")

Task Processing Order Times:

| Process |
| --- |
| process(O11, M1, 0) |
| process(O21, M2, 10) |
| process(O12, M2, 50) |
| process(O22, M1, 50) |
| process(O31, M1, 85) |
| process(O32, M2, 125) |


<br>

Job Completion Times

| Job | Completion Time |
| --- | --- |
| J1 | 75 |
| J2 | 85 |
| J3 | 145 |


Makespan: 145

### Question 5: SPT vs FSFS Comparison

The orderings and completion times are substantially different.

This doesn't mean that one algorithm is better than the other. A different arrival time would have different results.

SPT would work best when the tasks are all of similar effort, and the availability of machines is high - overload (not having an available machine) should be rare. Starving expensive tasks is a pretty standard response to overload conditions. Rejecting expensive work to process more total requests is (in practice) a good heuristic.

FCFS is a good general choice, but will perform poorly under overload. Jobs typically have a maximum
allowed completion time, and FCFS will allow _all_ jobs to exceed that time when overloaded. SPT will sacrifice one job to make the most progress through the queue.

Typically, I would recommend using FCFS until the system becomes overloaded (unable to meet service agreements), then switch to SPT (or reject oldest waiting task) for the overloaded machines.

### Question 6: Alternative methods

#### Static backlog

If the work is static, then it is a search problem.

We can map this to a graph search:

1) make each operation a node
2) connect each node to every other node -> (starting bidirectional)
3) remove backlinks from dependent nodes (e.g. delete O22 -> O21)
4) every edge has a cost representing the next node's start time (remaining cost of parent task + remaining time to machine availability)
6) starting from the root tasks (O11, O21, O31)
7) run A* on the graph, where

* g(x) is current makespan (including machine idleness)
* h(x) is the minimum workspan of the remaining nodes, assuming full utilization of each machine - the makespan's lower bound.

Note: The next hop's edge cost for g(n) will need to be recalculated to obtain the new start time each iteration.

Note: Since A* search is walking the task graph, but searching the set of process orderings, the cost of a transition depends on the history of the path, and nodes can't be pruned the same way it can with Euclidean maps. Beam and Bound may be possible, with the max workspan being the sum of all remaining task lengths - sequential ordering of tasks.

Since f(x) = g(x) + h(x), the algorithm is optimizing workspan.

#### Dynamic backlog

Here we need to define a fitness function. While the fitness function for static backlog was workspan, with genetic programming we can optimize for more things, or even combine fitness functions.

Examples:

* fitness = min idle workers     (min capital/labour cost)
* fitness = min total execution  (workspan)
* fitness = total value realized (overload handling)

The algorithm extends the problem to support multiple queues, but will not consider fitness functions other than workspan. Routing is only necessary if machines have multiple non-overlapping sets of capabilities. If machines are interchangeable (same capabilities, same location) with peers, the set can be represented by a single queue and the problem collapses to sequencing. The queue controls access to a set of machines all having the same set of capabilities.

The genetic programming solutions in the course separated routing from sequencing, first deciding where a task should go and then using another algorithm to learn task orderings on a machine.

The subset of queues which can process the task are selected, and the routing rule is used to score the task against each queue. The queue reporting the highest score is chosen and the task added to that queue. When a machine becomes "free", a sequencing rule is run on the controlling queue, with the highest scored operation selected.

The initial variables would be:

* m_wait_t - remaining wait time on the machine
* q_size - number of tasks in the machine's backlog
* q_length_t - total time of operations in queue
* op_wait_t - time required to run _this_ operation
* job_length_t - total time remaining for _this_ job
* job_length_op - total operations remaining for _this_ job

The operators would be:

* math - *,/,+,-
* grouping - min, max

A member of the population is the pair of {routing, sequencing} equations. Fitness will be measured against a set of recorded job arrivals and dependencies, replaying them against the rules and measuring workspan. Then standard crossover would be used, swapping subtrees with other rules. Crossover should also have the possibility of swapping the entire rule with another pair. Mutation would have the possibility of swapping the operation to another operation.

These mutations are safe because the operators all accept two numeric values and return a numeric value.


## Neural Networks



### Task A: Linearly Separable Data

**Code**: perceptron


| Model | Epoch | Accuracy |
| --- | --- | --- |
| Separable | 1 | 0.525 |
| Separable | 5 | 0.625 |
| Separable | 10 | 0.8 |
| Separable | 15 | 0.825 |
| Separable | 20 | 0.75 |
| Separable | 50 | 0.8 |
| Separable | 60 | 0.65 |
| Separable | 80 | 0.675 |
| Separable | 100 | 0.75 |
| Separable | 120 | 0.775 |
| Separable | 150 | 0.575 |
| Separable | 200 | 0.625 |


Increasing the number of epochs improves the accuracy until the perceptron overfits the training data. Then the performance starts to degrade. The choice of learning rate has an impact on the performance as well. A training rate of 1 worked remarkably well, finding the best performance at 10 epochs, and then overfitting. 0.1 was very unreliable, it had a hard time working down to the error minimum. 0.01 worked well, showing gradual improvement until it started overfitting. The best performing model is somewhere between 15 and 20 epochs with a training rate of 0.01. Using a validation set should allow the minimum to be found while avoiding overfitting.


### Task B: Non-Linearly Separable Data

**Code**: perceptron_b.py


| Model | Epoch | Accuracy |
| --- | --- | --- |
| Non-Separable | 1 | 0.4633333333333333 |
| Non-Separable | 5 | 0.5233333333333333 |
| Non-Separable | 10 | 0.6566666666666666 |
| Non-Separable | 15 | 0.7766666666666666 |
| Non-Separable | 20 | 0.7366666666666667 |
| Non-Separable | 50 | 0.6566666666666666 |
| Non-Separable | 60 | 0.6566666666666666 |
| Non-Separable | 80 | 0.6566666666666666 |
| Non-Separable | 100 | 0.6933333333333334 |
| Non-Separable | 120 | 0.6566666666666666 |
| Non-Separable | 150 | 0.6566666666666666 |
| Non-Separable | 200 | 0.6566666666666666 |

This perceptron was unable to separate the data, regardless of the number of epochs, or the training rate. It peaked at about 0.77. Plotting the test data, a single line would properly classify one class, and 50% of the other. If the classes were equal sized, that would represent an accuracy of ~0.75, which matches the training result.

### Task C: Multilayer Perceptron

**Code**: multi_layer_perceptron.py --max_epoch=300

Hyperparameters:

* EPOCHS = 300
* HIDDEN_LAYER_SIZES = (100,)
* LEARNING_RATE = 0.01
* ACTIVATION_FUNCTION = RelU

The model has a single hidden layer of 100 neurons. It produced a model with an accuracy of 0.95. It was able to accurately capture the structure of the data. Using multiple perceptrons allows more complex structures to be learned. The set of hidden neurons forms an interference pattern over the training points via the activation function. This allows each neuron to linearly subdivide the space, and for their probability distributions to constructively interfere and classify points.

### Task D: Activation Functions

| Activation Function | Epoch | Sizes | Accuracy |
| --- | --- | --- | --- |
| identity | 800 | (100,) | 0.763 |
| logistic | 800 | (100,) | 0.757 |
| tanh | 800 | (100,) | 0.940 |
| relu | 800 | (100,) | 0.950 |
| identity | 800 | (5, 2) | 0.757 |
| logistic | 800 | (5, 2) | 0.750 |
| tanh | 800 | (5, 2) | 0.940 |
| relu | 800 | (5, 2) | 0.943 |
| identity | 800 | (10, 5) | 0.760 |
| logistic | 800 | (10, 5) | 0.937 |
| tanh | 800 | (10, 5) | 0.930 |
| relu | 800 | (10, 5) | 0.947 |
| identity | 800 | (16, 8) | 0.763 |
| logistic | 800 | (16, 8) | 0.923 |
| tanh | 800 | (16, 8) | 0.937 |
| relu | 800 | (16, 8) | 0.957 |
| identity | 800 | (20, 10) | 0.783 |
| logistic | 800 | (20, 10) | 0.933 |
| tanh | 800 | (20, 10) | 0.930 |
| relu | 800 | (20, 10) | 0.953 |
| identity | 800 | (20, 10, 5) | 0.767 |
| logistic | 800 | (20, 10, 5) | 0.937 |
| tanh | 800 | (20, 10, 5) | 0.957 |
| relu | 800 | (20, 10, 5) | 0.960 |
| identity | 800 | (5, 10) | 0.763 |
| logistic | 800 | (5, 10) | 0.760 |
| tanh | 800 | (5, 10) | 0.940 |
| relu | 800 | (5, 10) | 0.940 |



RelU and TanH tended to perform better. However, with different layer configurations, logistic (sigmoid) came close. Sigmoid had problems converging in the default 300 epochs. At least 800 (between 750 and 800) epochs were required for Sigmoid to always converge in the test networks. RelU required 325 epochs, and TanH required 328 epochs to converge in all tested networks.

I ran tests with multiple network shapes. with the default network (100,), RelU and TanH performed roughly equivalently. Going to a two layer network allowed Logistic to catch up. Interestingly, a deeper network allows fewer total weights to be used and achieves better performance.

* (100,) = 100 * 3 (f1, f2, bias) = 300 weights
* (16,8) = 16 * 3 + 8 * 17 = 184 weights

RelU performed best. It won't be because of vanishing gradient, the single layer demonstrated the performance difference. It would have to be due to other features of RelU.

RelU has two other features, first, it passes all negative values as 0. This will allow neurons to only positively interfere. This will probably keep a neuron focused on a single task, instead of trying to optimize to find something and avoid other things.

Second, RelU is unbounded on the positive side. This allows a subset of neurons that divide the space well to do a lot of the work, allowing other neurons to find smaller components. For example, one neuron can divide the dataset and achieve 75% accuracy (from Task B). TahH and Logistic have upper and lower bounds. Sigmoid is asymptotic on approach to zero, so it is unable to properly indicate "no effect". TanH is able to encode both a zero and a negative number, allowing it to both indicate "no effect" and destructively interfere.

The choice of activation function will affect the cost to train (in convergence and total execution cost), and the final accuracy of the model. RelU is the cheapest to calculate back propagation with, and produces the best results. With this data, RelU is the best choice.





